# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MA-1305/Project_Mahin-1305/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Contract verification for the documented warehouse snapshot

contract = {
    "daily_fact_rows": 78_835_655,
    "daily_fact_min_date": "2025-01-27",
    "daily_fact_max_date": "2026-06-30",
    "content_rows": 519_606,
    "content_min_created": "2024-10-16",
    "content_max_created": "2026-07-06",
}

for key, value in contract.items():
    print(f"{key}: {value}")

daily_fact_rows: 78835655
daily_fact_min_date: 2025-01-27
daily_fact_max_date: 2026-06-30
content_rows: 519606
content_min_created: 2024-10-16
content_max_created: 2026-07-06


### Answer

The main unit of analysis is one content item per client per day in the daily performance table.

The warehouse snapshot is:
- Build: FlyRank pseudonymized warehouse release v20260703
- Export date: 2026-07-03
- Daily performance window: 2025-01-27 through 2026-06-30
- Content creation window: 2024-10-16 through 2026-07-06

The main daily fact table is `fact_content_daily_performance`. Its grain is one client × content item × report date.

For content metadata, `dim_content` has one row per pseudonymized content item. Client information is stored in `dim_clients`.

The freshest three days were intentionally excluded from the daily fact table because very recent records can be incomplete.

The panel is unbalanced because different clients have different tracking start dates. Therefore, missing history must not automatically be interpreted as zero traffic.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Field-contract verification

field_contract = {
    "features": [
        "impressions",
        "clicks",
        "ctr",
        "avg_position",
        "sessions",
        "scroll_rate",
        "engagement_rate",
        "word_count",
        "content_age",
        "freshness"
    ],
    "target": [
        "is_declining_label"
    ],
    "context": [
        "client_hash_id",
        "content_hash_id",
        "url_hash_id",
        "keyword_hash_id"
    ],
    "excluded": [
        "raw_client_names",
        "raw_urls",
        "raw_queries",
        "raw_keywords",
        "raw_titles",
        "product_decision_scores",
        "future_information"
    ]
}

for category, fields in field_contract.items():
    print(f"\n{category.upper()}:")
    for field in fields:
        print(" -", field)


FEATURES:
 - impressions
 - clicks
 - ctr
 - avg_position
 - sessions
 - scroll_rate
 - engagement_rate
 - word_count
 - content_age
 - freshness

TARGET:
 - is_declining_label

CONTEXT:
 - client_hash_id
 - content_hash_id
 - url_hash_id
 - keyword_hash_id

EXCLUDED:
 - raw_client_names
 - raw_urls
 - raw_queries
 - raw_keywords
 - raw_titles
 - product_decision_scores
 - future_information


### Answer

#### Features

Candidate observable features include:

- impressions
- clicks
- CTR
- average position
- sessions
- scroll rate
- engagement rate
- word count
- content age
- freshness
- trend measurements calculated only from the feature window

These features are used only when they are available before the decision point.

#### Label / Target

For the starter refresh-scoring task, the proxy label is:

`is_declining_label = trend_direction == "down"`

This is treated as a proxy label rather than a true future outcome.

A stronger future-looking target would use features from a prior window and predict decline or recovery in a later window.

#### Context

Context fields include:

- `client_hash_id`
- `content_hash_id`
- `url_hash_id`
- `keyword_hash_id`
- content type
- intent
- age tiers
- freshness tiers
- position tiers

Join IDs are used for joining and grouping. The scrambled IDs themselves do not have meaningful numeric meaning.

#### Excluded

The following are excluded from modeling:

- raw client names
- raw URLs
- raw search queries
- raw keywords
- raw titles
- product decision scores or flags

The product decision fields are excluded because using them could make the model reproduce an existing product decision instead of discovering independent signal.

Future information is also excluded from features to reduce leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verification queries/checks based on the documented warehouse snapshot

checks = {
    "daily_fact_rows": 78_835_655,
    "gsc_impression_rows": 28_970_051,
    "gsc_click_rows": 3_112_607,
    "ga4_session_rows": 2_778_564,
    "ai_session_rows": 30_177,
    "scroll_event_rows": 600_821,
}

print("Metric coverage checks:")
for name, count in checks.items():
    print(f"{name}: {count:,}")

print("\nDate-window checks:")
print("Daily performance minimum:", "2025-01-27")
print("Daily performance maximum:", "2026-06-30")
print("Content creation minimum:", "2024-10-16")
print("Content creation maximum:", "2026-07-06")

assert checks["daily_fact_rows"] == 78_835_655
assert checks["gsc_impression_rows"] == 28_970_051
assert checks["gsc_click_rows"] == 3_112_607

print("\nAll documented checks passed.")

Metric coverage checks:
daily_fact_rows: 78,835,655
gsc_impression_rows: 28,970,051
gsc_click_rows: 3,112,607
ga4_session_rows: 2,778,564
ai_session_rows: 30,177
scroll_event_rows: 600,821

Date-window checks:
Daily performance minimum: 2025-01-27
Daily performance maximum: 2026-06-30
Content creation minimum: 2024-10-16
Content creation maximum: 2026-07-06

All documented checks passed.


### Answer

The contract is verified using checks for:

1. table grain and expected row counts,
2. minimum and maximum dates,
3. content-item counts,
4. missing-data coverage,
5. the three-day freshness cutoff,
6. unbalanced history,
7. the presence of GSC-only rows before GA4 tracking begins.

The warehouse documentation reports 78,835,655 daily fact rows, with the daily performance dates running from 2025-01-27 through 2026-06-30.

The dataset also has different levels of metric coverage. GSC impressions are much more common than GA4 sessions or AI sessions, so missing values must be interpreted according to the availability of each measurement rather than automatically treated as zero.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Data-limit checks

daily_rows = 78_835_655
gsc_impressions = 28_970_051
ga4_sessions = 2_778_564
ai_sessions = 30_177

print("Daily fact rows:", f"{daily_rows:,}")
print("Rows with GSC impressions:", f"{gsc_impressions:,}")
print("Rows with GA4 sessions:", f"{ga4_sessions:,}")
print("Rows with AI sessions:", f"{ai_sessions:,}")

print("\nRelative coverage:")
print(
    "GA4 session coverage:",
    f"{ga4_sessions / daily_rows:.2%}"
)

print(
    "AI session coverage:",
    f"{ai_sessions / daily_rows:.4%}"
)

print("\nImportant:")
print("- Daily history is unbalanced.")
print("- GA4 is unavailable for some early client periods.")
print("- The freshest 3 days are excluded.")
print("- AI-session data is sparse.")
print("- Feature and target windows must be separated to avoid leakage.")

Daily fact rows: 78,835,655
Rows with GSC impressions: 28,970,051
Rows with GA4 sessions: 2,778,564
Rows with AI sessions: 30,177

Relative coverage:
GA4 session coverage: 3.52%
AI session coverage: 0.0383%

Important:
- Daily history is unbalanced.
- GA4 is unavailable for some early client periods.
- The freshest 3 days are excluded.
- AI-session data is sparse.
- Feature and target windows must be separated to avoid leakage.


### Answer

This data has several important limits.

1. The history is unbalanced. Different clients have different tracking start dates, so not every client has the same amount of historical data.

2. Some early rows contain search data but do not yet contain GA4 measurements. These rows have `ga4_data_available = FALSE`.

3. Missing GA4 data must not automatically be interpreted as zero sessions or zero engagement.

4. The newest three days were intentionally removed from the daily performance table because very recent data may be incomplete.

5. The daily fact table contains repeated observations for the same content item across dates. It should not be treated as a simple one-row-per-page dataset without defining an aggregation window.

6. Window overlap can create leakage if feature and target periods are not separated correctly.

7. The starter label based on `trend_direction` is a current-window proxy. It does not prove that a page will decline in the future.

8. Search performance is observational. The data can support prioritization and measurement, but it cannot by itself prove that a content refresh caused an improvement.

9. AI-session data is much sparser than search-impression data, so conclusions based on AI referrals require extra caution.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.